In [ ]:
# MLP with scikit-fingerprints
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader, TensorDataset

from skfp.fingerprints import ECFPFingerprint, MACCSFingerprint, TopologicalTorsionFingerprint
from skfp.preprocessing import MolFromSmilesTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. Load data
# ══════════════════════════════════════════════════════════════════════════════
DATA_DIR = "tasks-2026/task1"

df_train = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_train.parquet")
df_test = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_test_empty.parquet")

label_cols = [c for c in df_train.columns if c.startswith("class_")]
num_classes = len(label_cols)

print(f"Train: {len(df_train)} molecules, {num_classes} classes")
print(f"Test:  {len(df_test)} molecules")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. Compute fingerprints with scikit-fingerprints
#    Concatenate ECFP (2048), MACCS (167), TopologicalTorsion (2048)
# ══════════════════════════════════════════════════════════════════════════════
all_smiles = pd.concat([df_train["SMILES"], df_test["SMILES"]], ignore_index=True).tolist()

mol_transformer = MolFromSmilesTransformer()
all_mols = mol_transformer.transform(all_smiles)

ecfp = ECFPFingerprint(fp_size=2048, radius=2, n_jobs=-1)
maccs = MACCSFingerprint(n_jobs=-1)
torsion = TopologicalTorsionFingerprint(fp_size=2048, n_jobs=-1)

print("Computing ECFP...")
fp_ecfp = ecfp.transform(all_mols)
print("Computing MACCS...")
fp_maccs = maccs.transform(all_mols)
print("Computing TopologicalTorsion...")
fp_torsion = torsion.transform(all_mols)

X_all = np.hstack([fp_ecfp, fp_maccs, fp_torsion]).astype(np.float32)
print(f"Total fingerprint dim: {X_all.shape[1]}")

n_train = len(df_train)
X_train_full = X_all[:n_train]
X_test = X_all[n_train:]

Y_train_full = df_train[label_cols].values.astype(np.float32)

print(f"X_train: {X_train_full.shape}, X_test: {X_test.shape}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. Train/val split & dataloaders
# ══════════════════════════════════════════════════════════════════════════════
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_full, Y_train_full, test_size=0.1, random_state=42
)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val))
test_ds = TensorDataset(torch.from_numpy(X_test))

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. MLP Model
# ══════════════════════════════════════════════════════════════════════════════
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


INPUT_DIM = X_train.shape[1]
HIDDEN_DIM = 512
DROPOUT = 0.3

model = MLP(INPUT_DIM, HIDDEN_DIM, num_classes, DROPOUT).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Input dim: {INPUT_DIM}")
print(f"Model parameters: {total_params:,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. Loss — BCE with pos_weight
# ══════════════════════════════════════════════════════════════════════════════
pos_freq = Y_train_full.mean(axis=0)
pos_freq = np.clip(pos_freq, 0.001, 0.999)
pos_weight = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float).to(device)
pos_weight = torch.clamp(pos_weight, max=20.0)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Pos weight range: [{pos_weight.min():.2f}, {pos_weight.max():.2f}]")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. Evaluation
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            x, y = batch[0].to(device), batch[1].to(device)
            logits = model(x)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.cpu().numpy())
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    all_preds = (all_probs >= threshold).astype(float)

    f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return f1_micro, f1_macro

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. Training
# ══════════════════════════════════════════════════════════════════════════════
LR = 1e-3
EPOCHS = 60
PATIENCE = 10

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-6)

best_f1 = 0
patience_counter = 0
history = {"train_loss": [], "val_f1_micro": [], "val_f1_macro": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    n_samples = 0
    for batch in train_loader:
        x, y = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n_samples += x.size(0)

    scheduler.step()
    avg_loss = total_loss / n_samples

    f1_micro, f1_macro = evaluate(model, val_loader)

    history["train_loss"].append(avg_loss)
    history["val_f1_micro"].append(f1_micro)
    history["val_f1_macro"].append(f1_macro)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | "
          f"F1-micro: {f1_micro:.4f} | F1-macro: {f1_macro:.4f} | LR: {current_lr:.2e}")

    if f1_micro > best_f1:
        best_f1 = f1_micro
        patience_counter = 0
        torch.save(model.state_dict(), "best_model_mlp_fp.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest validation F1-micro: {best_f1:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8. Training curves
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"])
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_f1_micro"], label="F1-micro")
axes[1].plot(history["val_f1_macro"], label="F1-macro")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves_mlp_fp.png", dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9. Generate test predictions & submission
# ══════════════════════════════════════════════════════════════════════════════
model.load_state_dict(torch.load("best_model_mlp_fp.pt", weights_only=True))
model.eval()

all_preds = []
with torch.no_grad():
    for batch in test_loader:
        x = batch[0].to(device)
        logits = model(x)
        preds = (torch.sigmoid(logits) >= 0.5).int()
        all_preds.append(preds.cpu().numpy())

all_preds = np.vstack(all_preds)

submission = pd.DataFrame(all_preds, columns=label_cols)
submission.insert(0, "mol_id", df_test["mol_id"].values)
submission.insert(1, "SMILES", df_test["SMILES"].values)

print(f"Submission shape: {submission.shape}")
submission.to_parquet("chebi_submission_mlp_fp.parquet", index=False)
print("Saved to chebi_submission_mlp_fp.parquet")